In [2]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

# ====================== 1. 讀取資料 ======================
df = pd.read_excel('591_實價登錄整理.xlsx')

print("資料筆數:", len(df))
print(df.head())

# ====================== 2. 資料準備 ======================
df['ln_total_price'] = np.log(df['成交總價_含車位_萬元'])

# ====================== 3. 基礎模型（先不放距離變數） ======================
formula1 = """
ln_total_price ~ 建坪_含車位 + 屋齡 + 屋齡平方 + 中高樓層虛擬變數 + 有車位虛擬變數
"""

model1 = smf.ols(formula=formula1, data=df).fit()

print("\n=== 基礎模型結果 ===")
print(model1.summary())

# ====================== 4. VIF 多重共線性檢定 ======================
features = ['建坪_含車位', '屋齡', '屋齡平方', '中高樓層虛擬變數', '有車位虛擬變數']

X = df[features].copy()
X = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data["變數"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\n=== VIF 檢定 ===")
print(vif_data)

# ====================== 5. 對勘估標的進行預測 ======================
# 我們的標的物：67.44坪，屋齡9.31，中高樓層=1，有車位=1
subject = pd.DataFrame({
    '建坪_含車位': [67.44],
    '屋齡': [9.31],
    '屋齡平方': [9.31**2],
    '中高樓層虛擬變數': [1],
    '有車位虛擬變數': [1]
})

pred_ln = model1.predict(subject)
pred_price = np.exp(pred_ln[0])

print("\n=== 對勘估標的預測 ===")
print(f"預測總價：{pred_price:,.0f} 萬元")
print(f"預測單價：{pred_price / 67.44:.2f} 萬/坪")

資料筆數: 46
   Case_ID    成交年月        成交日期  成交總價_含車位_萬元  建坪_含車位  樓層  總樓層    屋齡     屋齡平方  \
0  7256601  115-02  2026-02-01         1880    67.4   5    9  9.31  86.6761   
1  7121628  114-10  2025-10-13          670    25.5   1    9  9.00  81.0000   
2  7091875  114-06  2025-06-08         1870    65.4   4    9  8.66  74.9956   
3  6933352  114-02  2025-02-17         1870    71.7   9    9  8.35  69.7225   
4  6334946  113-01  2024-01-09         2838    97.0   1    9  7.24  52.4176   

   有車位虛擬變數  中高樓層虛擬變數                地址  到東華附小距離(google)  到球崙公園距離(google)  \
0        1         1    中美路111號 | 5樓之1              700             1300   
1        1         0  中美路111號之6 | 1,2樓              700             1300   
2        1         1    中美路111號 | 4樓之3              700             1300   
3        1         1    中美路111號 | 9樓之1              700             1300   
4        1         0  中美路111號之7 | 1,2樓              700             1300   

   到門諾距離(google)  
0            750  
1            750  
2 

In [3]:
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import pandas as pd

# 確保資料已讀取（df 存在）
# ====================== 1. 定義新模型的自變數 (拿掉屋齡平方) ======================
X_new_vars = ['建坪_含車位', '屋齡', '中高樓層虛擬變數', '有車位虛擬變數']

X_new = df[X_new_vars]
X_new = sm.add_constant(X_new) # 加入常數項
y = df['ln_total_price']

# ====================== 2. 跑全新迴歸模型 ======================
model_new = sm.OLS(y, X_new).fit()

print("="*20 + " 修正後模型結果 (已移除屋齡平方) " + "="*20)
print(model_new.summary())

# ====================== 3. 計算修正後的 VIF ======================
vif_data_new = pd.DataFrame()
vif_data_new["變數"] = X_new.columns
vif_data_new["修正後 VIF"] = [variance_inflation_factor(X_new.values, i) for i in range(X_new.shape[1])]

print("\n" + "="*20 + " 修正後 VIF 檢定 " + "="*20)
print(vif_data_new.to_string(index=False))

# ====================== 4. 新模型對勘估標的之預測 ======================
# 假設你的勘估標的特徵如下（請確認是否與你原本預測的一致）：
# 這裡以第一筆資料作為示範，你可以根據需要調整數值
target_data = {
    'const': 1.0,
    '建坪_含車位': 65.4,          # 請填入勘估標的建坪
    '屋齡': 8.66,               # 請填入勘估標的屋齡
    '中高樓層虛擬變數': 1,       # 是=1, 否=0
    '有車位虛擬變數': 1          # 是=1, 否=0
}
target_df = pd.DataFrame([target_data])

# 預測
pred_ln_price = model_new.predict(target_df)[0]
pred_total_price = np.exp(pred_ln_price) # 因為 y 取了 ln，這裡要用 exp 還原總價
pred_unit_price = pred_total_price / target_data['建坪_含車位']

print("\n" + "="*20 + " 修正後模型之勘估標的預測 " + "="*20)
print(f"預測總價：{pred_total_price:,.0f} 萬元")
print(f"預測單價：{pred_unit_price:.2f} 萬/坪")

==================== 修正後模型結果 (已移除屋齡平方) ====================
                            OLS Regression Results                            
Dep. Variable:         ln_total_price   R-squared:                       0.880
Model:                            OLS   Adj. R-squared:                  0.868
Method:                 Least Squares   F-statistic:                     75.05
Date:                Mon, 01 Jun 2026   Prob (F-statistic):           2.60e-18
Time:                        13:43:01   Log-Likelihood:                 50.000
No. Observations:                  46   AIC:                            -90.00
Df Residuals:                      41   BIC:                            -80.86
Df Model:                           4                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
-----------------------------------------------------------------------

In [5]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

# 讀取資料
df = pd.read_excel('591_實價登錄整理.xlsx')

print("資料筆數:", len(df))

# 資料準備
df['ln_total_price'] = np.log(df['成交總價_含車位_萬元'])

# ====================== 新模型：樓層改為連續變數 ======================
formula = """
ln_total_price ~ 建坪_含車位 + 屋齡 + 中高樓層虛擬變數 + 有車位虛擬變數 + 樓層
"""

model = smf.ols(formula=formula, data=df).fit()

print("\n=== 樓層改為連續變數後的模型結果 ===")
print(model.summary())

# VIF 檢定
features = ['建坪_含車位', '屋齡', '中高樓層虛擬變數', '有車位虛擬變數', '樓層']
X = df[features].copy()
X = sm.add_constant(X)
vif_data = pd.DataFrame()
vif_data["變數"] = X.columns
vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]
print("\n=== VIF 檢定 ===")
print(vif_data)

# 對勘估標的預測 (樓層假設為5樓)
subject = pd.DataFrame({
    '建坪_含車位': [67.44],
    '屋齡': [9.31],
    '中高樓層虛擬變數': [1],
    '有車位虛擬變數': [1],
    '樓層': [5]
})

pred_ln = model.predict(subject)
pred_price = np.exp(pred_ln[0])

print("\n=== 對勘估標的預測 ===")
print(f"預測總價：{pred_price:,.0f} 萬元")
print(f"預測單價：{pred_price / 67.44:.2f} 萬/坪")

資料筆數: 46

=== 樓層改為連續變數後的模型結果 ===
                            OLS Regression Results                            
Dep. Variable:         ln_total_price   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.898
Method:                 Least Squares   F-statistic:                     79.87
Date:                Mon, 01 Jun 2026   Prob (F-statistic):           9.85e-20
Time:                        13:48:08   Log-Likelihood:                 56.382
No. Observations:                  46   AIC:                            -100.8
Df Residuals:                      40   BIC:                            -89.79
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept      6.15

In [7]:
import pandas as pd
import numpy as np
import statsmodels.formula.api as smf
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
import matplotlib.pyplot as plt
import seaborn as sns

# 1. 讀取資料
df = pd.read_excel('591_實價登錄整理.xlsx')
print("資料筆數:", len(df))

# 2. 資料準備（依變數對數轉換）
df['ln_total_price'] = np.log(df['成交總價_含車位_萬元'])

# 💡 關鍵計量實驗設定：
# 因為同社區到東華附小距離皆為 700 公尺（常數），直接放入會與 Intercept 產生「完全共線」而遭 Python 自動剔除。
# 為使模型能強制估計並在 VIF 報表中暴露出「無限大(inf)」的共線性黑洞，我們加入微幅的計量擾動（或直接填入真實常數 700）。
df['到東華附小距離_公尺'] = 700.0  

# ==============================================================================
# 實證模型 A：最終優化模型（包含樓層連續變數，不含完全共線變數）
# ==============================================================================
formula_final = """
ln_total_price ~ 建坪_含車位 + 屋齡 + 中高樓層虛擬變數 + 有車位虛擬變數 + 樓層
"""
model_final = smf.ols(formula=formula_final, data=df).fit()

print("\n" + "="*80)
print("=== 模型一：最終優化模型結果 (解鎖九成解釋力) ===")
print("="*80)
print(model_final.summary())

# ==============================================================================
# 實證模型 B：進階實驗模型（強行納入「到東華附小距離」進行共線壓力測試）
# ==============================================================================
# 註：此處為了強迫 statsmodels 進行矩陣運算並產出 VIF 警告，我們對常數加上極微小的擾動項（0.0001），模擬完全共線。
df['到東華附小距離_壓力測試'] = df['到東華附小距離_公尺'] + np.random.normal(0, 0.0001, len(df))

formula_exp = """
ln_total_price ~ 建坪_含車位 + 屋齡 + 中高樓層虛擬變數 + 有車位虛擬變數 + 樓層 + 到東華附小距離_壓力測試
"""
model_exp = smf.ols(formula=formula_exp, data=df).fit()

print("\n" + "="*80)
print("=== 模型二：強行加入「到東華附小距離」後的壓力測試結果 ===")
print("="*80)
print(model_exp.summary())

# ==============================================================================
# VIF 多重共線性檢定 (針對實驗模型)
# ==============================================================================
features = ['建坪_含車位', '屋齡', '中高樓層虛擬變數', '有車位虛擬變數', '樓層', '到東華附小距離_壓力測試']
X = df[features].copy()
X = sm.add_constant(X)

vif_data = pd.DataFrame()
vif_data["特徵自變數"] = X.columns
vif_data["VIF 統計值"] = [variance_inflation_factor(X.values, i) for i in range(X.shape[1])]

print("\n" + "="*80)
print("=== 實驗模型之 VIF 多重共線性檢定報表 ===")
print("="*80)
print(vif_data.to_string(index=False))
print("\n💡 計量學註記：『到東華附小距離』與『const(截距)』之 VIF 暴增至無限大(或數萬)，")
print("   鐵證了同社區微觀估價下，區位常數變數存在『完全多重共線性』，驗證 HPM 之理論限制。")

# ==============================================================================
# 對勘估標的最終預測 (基於穩定度最高、無完全共線之模型一)
# ==============================================================================
subject = pd.DataFrame({
    '建坪_含車位': [67.44],
    '屋齡': [9.31],
    '中高樓層虛擬變數': [1],
    '有車位虛擬變數': [1],
    '樓層': [5]
})

pred_ln = model_final.predict(subject)
pred_price = np.exp(pred_ln[0])

print("\n" + "="*80)
print("=== 最終優化模型對勘估標的（美崙之星 5樓之1）之價值預測 ===")
print("="*80)
print(f"▶ HPM 模型預估客觀總價：{pred_price:,.2f} 萬元")
print(f"▶ HPM 模型預估客觀單價：{pred_price / 67.44:.2f} 萬元 / 坪")
print("="*80)

資料筆數: 46

=== 模型一：最終優化模型結果 (解鎖九成解釋力) ===
                            OLS Regression Results                            
Dep. Variable:         ln_total_price   R-squared:                       0.909
Model:                            OLS   Adj. R-squared:                  0.898
Method:                 Least Squares   F-statistic:                     79.87
Date:                Mon, 01 Jun 2026   Prob (F-statistic):           9.85e-20
Time:                        14:35:52   Log-Likelihood:                 56.382
No. Observations:                  46   AIC:                            -100.8
Df Residuals:                      40   BIC:                            -89.79
Df Model:                           5                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
Intercept  